# V2 — Generacja RAW (Kaggle)

Pliki projektu: `/kaggle/input/datasets/kamilml/sources/` (wgraj cały folder `Steganography_benchmarks_V2/`)

**Kaggle nie cache'uje modeli** — zawsze pobiera z Hugging Face (`--platform kaggle`).

Wymaga: Internet + secret `HF_TOKEN`.

**Ważne — zapis wyników:** Kaggle kasuje `/kaggle/working` po zamknięciu/resecie sesji. Żeby nie stracić danych:
1. Po generacji pobierz `runs_backup_*.zip` z zakładki **Output** (prawy panel), albo
2. **Save Version → Quick Save** z włączonym *Save output*, albo
3. Ustaw `RESUME_RUN_DIR` i wznów przerwaną generację (pliki są dopisywane co zadanie).

In [3]:
import os
import shutil
from pathlib import Path

INPUT_DIR = Path('/kaggle/input/datasets/kamilml/sources1')
PROJECT_DIR = Path('/kaggle/working/Steganography_benchmarks_V2')

V2_FILES = [
    'generate_responses.py',
    'evaluate_responses.py',
    'eval_handlers.py',
    'common.py',
    'code_extract.py',
    'prompts.py',
    'model_runtime.py',
    'raw_store.py',
    'dummy_processor.py',
    'humaneval_subset_eval.py',
    'perplexity_metrics.py',
    'binoculars_scorer.py',
    'notebook_runner.py',
    'requirements.txt',
]

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
missing = [name for name in V2_FILES if not (INPUT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f'Brakuje w {INPUT_DIR}: {missing}')

for name in V2_FILES:
    shutil.copy2(INPUT_DIR / name, PROJECT_DIR / name)
    print(f'copied {name}')

os.chdir(PROJECT_DIR)
print('cwd:', Path.cwd())


copied generate_responses.py
copied evaluate_responses.py
copied eval_handlers.py
copied common.py
copied code_extract.py
copied prompts.py
copied model_runtime.py
copied raw_store.py
copied dummy_processor.py
copied humaneval_subset_eval.py
copied perplexity_metrics.py
copied binoculars_scorer.py
copied notebook_runner.py
copied requirements.txt
cwd: /kaggle/working/Steganography_benchmarks_V2


In [4]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.3/52.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.1 MB/s eta 0:00:00


In [5]:
import os
from kaggle_secrets import UserSecretsClient

if not os.getenv('HF_TOKEN'):
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

In [6]:
# --- KONFIGURACJA ---
TEST = 'humaneval'        # humaneval | perplexity | capacity | binoculars
MODEL = 'gemma'
THRESHOLD = 0.1          # uruchom komórkę 3×: 0.01, 0.05, 0.1
TOP_N = 16
MAX_NEW_TOKENS = 512
HUMANEVAL_TASKS = '74-163'  # zadania 74..163 (91 szt.); None = wszystkie 164
RESUME_RUN_DIR = None    # np. 'runs/2026-07-11_...' — wznów po przerwaniu

import shutil
import sys
from datetime import datetime
from pathlib import Path

from notebook_runner import run_live


def backup_runs() -> None:
    """Zip runs/ → /kaggle/working (Output tab). Uruchamiane też po błędzie."""
    runs = Path('runs')
    if not runs.exists():
        print('Brak folderu runs/ — nic do backupu.')
        return
    jsonl_files = list(runs.rglob('*.jsonl'))
    if not jsonl_files:
        print('Brak *.jsonl w runs/.')
        return
    for p in jsonl_files:
        lines = sum(1 for _ in p.open(encoding='utf-8'))
        print(f'  {p.relative_to(runs)}: {lines} linii')
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
    archive_base = out_dir / f'runs_backup_{ts}'
    archive = shutil.make_archive(str(archive_base), 'zip', runs)
    print(f'Backup zapisany: {archive}')
    print('Pobierz z zakładki Output (prawy panel) ZANIM zamkniesz sesję.')


print('cwd:', Path.cwd())
print('generate_responses.py:', Path('generate_responses.py').exists())

if RESUME_RUN_DIR:
    cmd = [sys.executable, 'generate_responses.py', '--run-dir', str(RESUME_RUN_DIR)]
else:
    cmd = [
        sys.executable,
        'generate_responses.py',
        '--model', MODEL,
        '--threshold', str(THRESHOLD),
        '--top-n', str(TOP_N),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--platform', 'kaggle',
        '--output-root', 'runs',
    ]
    # Stary skrypt z datasetu Kaggle nie ma --test (humaneval jest domyślny).
    if TEST != 'humaneval':
        cmd += ['--test', TEST]
    if HUMANEVAL_TASKS and TEST == 'humaneval':
        cmd += ['--humaneval-tasks', HUMANEVAL_TASKS]

print('cmd:', ' '.join(cmd))

try:
    run_live(cmd)
except Exception as exc:
    print(f'\nBłąd generacji: {exc}')
    print('Częściowe dane mogą być w runs/ — robię backup...')
finally:
    backup_runs()

print('\nPo pobraniu zipa oceń lokalnie: python run_evaluate.py <nazwa_runu>')


usage: generate_responses.py [-h] [--model MODEL] [--threshold THRESHOLD]
                             [--top-n TOP_N] [--max-new-tokens MAX_NEW_TOKENS]
                             [--seed SEED] [--humaneval-tasks HUMANEVAL_TASKS]
                             [--output-root OUTPUT_ROOT] [--run-dir RUN_DIR]
                             [--platform {colab,kaggle}]
                             [--model-cache-dir MODEL_CACHE_DIR] [--offline]
                             [--list-models]
generate_responses.py: error: unrecognized arguments: --test humaneval

Błąd generacji: Command '['python', '-u', 'generate_responses.py', '--test', 'humaneval', '--model', 'gemma', '--threshold', '0.1', '--top-n', '16', '--max-new-tokens', '512', '--platform', 'kaggle', '--output-root', 'runs', '--humaneval-tasks', '74-163']' returned non-zero exit status 2.
Częściowe dane mogą być w runs/ — robię backup...
Brak folderu runs/ — nic do backupu.

Po pobraniu zipa oceń lokalnie: python run_evaluate.py <nazwa_

In [7]:
# Uruchom ręcznie, jeśli sesja jeszcze żyje (np. po błędzie w innej komórce)
from pathlib import Path

runs = Path('runs')
if runs.exists():
    for p in sorted(runs.rglob('*.jsonl')):
        n = sum(1 for _ in p.open(encoding='utf-8'))
        print(f'{p}: {n} próbek')
else:
    print('Brak runs/ — sesja mogła zostać zresetowana, danych już nie ma.')

# backup_runs()  # odkomentuj, jeśli komórka generacji nie doszła do finally

Brak runs/ — sesja mogła zostać zresetowana, danych już nie ma.
